In [5]:
import pandas as pd
from course_utils.paths import get_data_dir

data_dir = get_data_dir()

customers   = pd.read_csv(data_dir / "customers.csv")
orders      = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")
products    = pd.read_csv(data_dir / "products.csv")

print(customers.shape, orders.shape, order_items.shape, products.shape)

(150, 6) (300, 5) (764, 5) (100, 4)


In [6]:
from course_utils.paths import get_project_root, get_data_dir

print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

프로젝트 루트 : C:\dev\llm-data-analysis-course\llm-data-analysis
데이터 폴더 : C:\dev\llm-data-analysis-course\llm-data-analysis\data\raw


In [7]:
import pandas as pd

orders = pd.read_csv("../../data/raw/orders.csv")

orders.head()

,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-07-02,card,completed
1,2,77,2025-09-17,naver_pay,cancelled
2,3,138,2026-01-14,bank_transfer,cancelled
3,4,57,2026-03-27,kakao_pay,cancelled
4,5,125,2026-02-15,card,cancelled


# STEP 2. 실제 컬럼명과 값부터 확인합니다

In [8]:
print(customers["city"].value_counts(dropna=False))
print(orders["order_status"].value_counts(dropna=False))

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64
order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


# STEP 3. 필요한 컬럼과 행을 선택합니다

In [9]:
customer_basic = customers[[
    "customer_id",
    "gender",
    "age",
    "city",
]]
print(customer_basic.head())
customer_over_30 = customers[
    customers["age"]>=30
]
print(customer_over_30.sort_values("customer_id").head())
city_customers = customers[
    customers["city"].isin(["서울","부산"])
]
print(city_customers.head(10))

   customer_id gender  age city
0            1      F   19   광주
1            2      F   32   대구
2            3      F   61   성남
3            4      F   55   울산
4            5      F   19   부산
   customer_id name gender  age city signup_date
1            2  김정호      F   32   대구  2025-12-28
2            3  이경수      F   61   성남  2024-08-07
3            4  조영호      F   55   울산  2026-06-08
5            6  김지원      F   32   성남  2026-08-22
6            7  이상현      F   53   인천  2025-02-06
    customer_id name gender  age city signup_date
4             5  이예원      F   19   부산  2024-11-08
8             9  송지민      M   69   서울  2025-12-14
14           15  장정식      M   69   서울  2026-07-30
15           16  강보람      M   52   부산  2024-09-17
29           30  이민재      F   32   서울  2023-09-08
39           40  박예준      M   23   서울  2024-07-28
41           42  장성호      M   56   부산  2025-11-15
44           45  김명자      F   61   부산  2023-10-21
47           48  김예은      F   47   서울  2025-05-27
53           5

In [10]:
products.sort_values(
    "price",
    ascending=False
).head(10)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


# STEP 4. line_total 파생 컬럼을 만듭니다

In [11]:
order_items = order_items.copy()
order_items["line_total"] = (
    order_items["quantity"]*order_items["unit_price"]
)

print(order_items.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


현재 line_total 전체 합계에는 주문 상태가 반영되지 않았으므로
완료 주문 기준 금액으로 해석하면 안 된다고 판단했다.


# STEP 5. orders와 병합하기 전에 키를 확인합니다

In [12]:
print(
    "orders.order_id 중복 수:",
    orders["order_id"].duplicated().sum()
)

orders.order_id 중복 수: 0


In [13]:
order_sales = order_items.merge(
    orders[["order_id",
           "customer_id",
           "order_date",
           "order_status"
           ]],
           on = "order_id",
           how="left",
           validate="many_to_one",
           indicator=True,
)
print(order_sales.info())

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   order_item_id  764 non-null    int64   
 1   order_id       764 non-null    int64   
 2   product_id     764 non-null    int64   
 3   quantity       764 non-null    int64   
 4   unit_price     764 non-null    int64   
 5   line_total     764 non-null    int64   
 6   customer_id    764 non-null    int64   
 7   order_date     764 non-null    str     
 8   order_status   764 non-null    str     
 9   _merge         764 non-null    category
dtypes: category(1), int64(7), str(2)
memory usage: 54.7 KB
None


병합이 잘 되었는지 검증하기 위해 '_merge' 컬럼을 넣음. 
indicator = True 이게 pandas가 병합 결과에 _merge라는 컬럼을 자동으로 추가해주는 명령어이다.

In [14]:
print("병합 전 행 수:", len(order_items))
print("병합 후 행 수:", len(order_sales))
print(order_sales["_merge"].value_counts())

병합 전 행 수: 764
병합 후 행 수: 764
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64


In [15]:
# 검증이 끝났으므로 _merge 제거. drop
order_sales = order_sales.drop(columns="_merge")

# STEP 6. 날짜를 변환하고 completed 주문만 선택

In [16]:
order_sales["order_date"] = pd.to_datetime(
    order_sales["order_date"],
    errors="coerce",
)

print(
    "날짜 변환 실패:",
    order_sales["order_date"].isna().sum()
)

날짜 변환 실패: 0


In [17]:
completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [18]:
completed_order_sales["order_status"].value_counts(dropna=False)

order_status
completed    474
Name: count, dtype: int64

In [19]:
completed_order_sales["order_month"] = (
    completed_order_sales["order_date"]
    .dt.to_period("M")
    .astype(str)
)

print(completed_order_sales.head())

    order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0               1         1         100         3      102000      306000   
1               2         1          87         5       25000      125000   
2               3         1           7         3      142000      426000   
3               4         1           9         3      193000      579000   
12             13         6          83         3       24000       72000   

    customer_id order_date order_status order_month  
0           123 2026-07-02    completed     2026-07  
1           123 2026-07-02    completed     2026-07  
2           123 2026-07-02    completed     2026-07  
3           123 2026-07-02    completed     2026-07  
12           87 2026-05-16    completed     2026-05  


# STEP 7. products를 연결하고 다시 검증합니다

In [20]:
completed_sales_items = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
print(completed_sales_items.dtypes)

order_item_id             int64
order_id                  int64
product_id                int64
quantity                  int64
unit_price                int64
line_total                int64
customer_id               int64
order_date       datetime64[us]
order_status                str
order_month                 str
product_name                str
category                    str
price                     int64
_merge                 category
dtype: object


In [21]:
print("병합 전:", len(completed_order_sales))
print("병합 후:", len(completed_sales_items))
print(completed_sales_items["_merge"].value_counts())

병합 전: 474
병합 후: 474
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64


In [22]:
products["product_id"].duplicated().sum()

np.int64(0)

In [23]:
completed_sales_items = completed_sales_items.drop(columns="_merge")

# STEP 8. 카테고리별·상품별 집계를 만듭니다

SyntaxError: invalid syntax (2909281078.py, line 1)